## **Sampling Search Algorithms in LLMs**

When an autoregressive model (like an Image Captioner or LLM) generates text, it outputs a probability distribution over the entire vocabulary at each step. A search algorithm determines how the model selects the next token from those probabilities to form a coherent sequence.

#### 1. The Baseline: Exhaustive Search (Brute Force)
Exhaustive search evaluates every possible combination of words across vocabulary $V$ for a sequence of length $T$.
* Total combinations: $|V|^T$ (e.g., $10,000^{20} \approx 10^{80}$)
* Computationally impossible. Practical decoding methods approximate this search using heuristics or sampling.

#### 2. Deterministic Search Strategies

#### Greedy Search (`argmax`)
At each step, greedy search chooses the single token with the highest predicted probability.
* Formula: $w_t = \arg\max_w P(w \mid w_{1:t-1})$
* Complexity: $O(T)$, minimal memory footprint.
* Strengths: Fastest possible decoding strategy.
* Limitations: Short-sighted. A high-probability initial word may lead to lower overall sequence probability later. Frequently leads to repetitive loops (e.g., *"a dog and a dog and a dog"*).

#### Beam Search
Beam search maintains a fixed number of candidate sequences (called the beam width, $B$, typically 3 to 5) at every step.
* Process:
  1. At step 1, keep the top $B$ most probable words.
  2. At step 2, expand all $B$ paths across the entire vocabulary ($B \times |V|$ paths).
  3. Compute cumulative sequence probabilities: $\prod_{t} P(w_t \mid w_{1:t-1})$.
  4. Prune the search tree back to the top $B$ candidates.
  5. Repeat until the `<end>` token is produced for all candidates.
* Strengths: Recovers globally high-probability sentences that greedy search misses.
* Limitations: $B\times$ slower than greedy. Tends to favor generic, conservative phrasing. Standard choice for factual tasks like translation and image captioning.

#### 3. Stochastic Sampling Strategies

#### Pure Random Sampling
Tokens are randomly drawn according to their output probability distribution.
* Limitations: The long tail of thousands of rare tokens (each with $<0.01\%$ probability) collectively carries enough cumulative mass that the model will eventually draw an irrelevant or ungrammatical word.

#### Temperature Sampling
Temperature ($T$) scales the raw logits $z_i$ prior to the softmax function:
$$P(w_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)}$$
* $T \to 0$ (e.g., 0.1 to 0.2): Sharpened distribution. Heavily favors top tokens; approaches greedy search.
* $T = 1.0$: Default model distribution.
* $T > 1.0$ (e.g., 1.2 to 1.5): Flattened distribution. Increases diversity and creativity, but elevates hallucination risk.

#### Top-$k$ Sampling
Restricts selection to the top $k$ highest-probability tokens (e.g., $k=50$). The remaining tokens are zeroed out, and probabilities are re-normalized before sampling.
* Strengths: Completely removes the long tail of low-probability words.
* Limitations: $k$ is static. If the context makes the next word unambiguous (99% probability for one token), top-$k$ still forces sampling across $k$ choices, occasionally introducing errors.

#### Top-$p$ Sampling (Nucleus Sampling)
Instead of a fixed count $k$, top-$p$ sets a cumulative probability threshold (e.g., $p = 0.90$). The candidate pool dynamically includes only the smallest set of top tokens whose cumulative probability reaches $p$.
* If the model is confident, the candidate pool automatically shrinks to 1 or 2 tokens.
* If the model is uncertain, the pool dynamically expands to include dozens of plausible words.
* Standard configuration for modern conversational LLMs (often paired with $T \approx 0.7$).

#### 4. Penalty Modifiers

#### Repetition Penalty
Divides the logit of any token that has already appeared in the generated sequence by a scalar penalty (e.g., 1.2). Suppresses redundant phrases and loop cycles.

#### Length Penalty
In beam search, raw cumulative probabilities decrease as sequences grow longer ($\prod p_i$). A length normalizer divides the score by $(\text{length})^\alpha$ (typically $\alpha = 0.6 \text{ to } 0.8$) so the model is not penalized for generating full-length sentences.

#### 5. Comparison Matrix

| Strategy | Search Type | Speed | Deterministic | Primary Use Case |
| :--- | :--- | :--- | :--- | :--- |
| **Greedy** | Heuristic Search | Fast | Yes | Low-latency inference, debugging |
| **Beam Search** | Heuristic Search | Moderate | Yes | Image captioning, translation, summarization |
| **Temperature** | Sampling Modifier | Fast | No | Controlling randomness level |
| **Top-$k$** | Truncated Sampling | Fast | No | Basic text generation |
| **Top-$p$ (Nucleus)**| Dynamic Sampling | Fast | No | Conversational LLMs, creative writing |

---

### **How TOP-K is different from Greedy Seach ???**

### Difference Between Greedy Search and Top-k Sampling

The primary difference between Greedy Search and Top-k Sampling is how they handle choice: **Greedy Search is deterministic and picks only the single highest-probability token**, while **Top-k Sampling is stochastic (randomized) and samples from a pool of the top $k$ candidates**.

#### 1. Core Mechanism Comparison

#### Greedy Search ($k = 1$)
* Strictly takes the token with the highest probability: $w_t = \arg\max P(w \mid w_{1:t-1})$.
* Candidate pool size is always 1.
* Output is 100% deterministic: feeding the exact same image or prompt will always yield the exact same caption.

#### Top-k Sampling ($k > 1$, e.g., $k = 50$)
* Sorts all vocabulary tokens by probability and keeps only the top $k$ candidates.
* Sets the probability of all other tokens to zero.
* Re-normalizes the remaining $k$ probabilities so they sum to 1.0.
* Randomly draws a token based on those relative weights (a weighted dice roll).
* Output is non-deterministic: the same input produces different, diverse variations across runs.

#### 2. Walkthrough Example

Assume the model predicts the following probabilities for the next word after *"A woman is standing on a"*:

| Token | Raw Probability | Greedy Search ($k=1$) | Top-k Sampling ($k=4$) |
| :--- | :--- | :--- | :--- |
| **"field"** | 45% | **Selected (100% of the time)** | 47.4% chance to be picked |
| **"beach"** | 30% | Ignored | 31.6% chance to be picked |
| **"street"** | 15% | Ignored | 15.8% chance to be picked |
| **"sidewalk"** | 5% | Ignored | 5.2% chance to be picked |
| *"elephant"* | 0.01% | Ignored | Discarded ($> k$) |
| *"guitar"* | 0.005% | Ignored | Discarded ($> k$) |

* **Greedy Search** will pick `"field"` every single time without exception.
* **Top-k Sampling** usually picks `"field"`, but roughly 3 out of 10 runs it will pick `"beach"`, and occasionally `"street"` or `"sidewalk"`. It will never pick `"elephant"` because it sits outside the top $k=4$.

#### 3. Strengths and Weaknesses

#### Greedy Search
* Strengths: Fastest possible inference; zero variance; ideal for factual tasks with a single clear answer.
* Weaknesses: Highly prone to repetitive loops (e.g., *"in a field in a field"*); misses globally optimal sentences because it never explores paths starting with the second or third best word.

#### Top-k Sampling
* Strengths: Eliminates repetitive loops; produces more natural, human-like variation; cuts off the dangerous long tail of gibberish words.
* Weaknesses: $k$ is static. If the model is 99% certain about a word (e.g., *"Statue of [Liberty]"*), Top-$k$ still forces the remaining $k-1$ options into the candidate pool, occasionally sampling an incorrect word.

#### 4. Summary Table

| Feature | Greedy Search | Top-k Sampling |
| :--- | :--- | :--- |
| **Selection Type** | Deterministic (`argmax`) | Stochastic (Weighted Random Sample) |
| **Candidate Pool** | Exactly 1 token | Fixed $k$ tokens (e.g., 20 to 50) |
| **Output Repeatability**| Identical every run | Different on every run |
| **Risk of Loops** | High (frequently repeats phrases) | Low (randomness breaks cycles) |
| **Risk of Errors** | Misses high-probability future paths | May pick a lower-ranked word when 1 word was obvious |
| **Relationship** | Top-k where $k = 1$ | Generalization of greedy search across $k$ choices |